# 03_preparacao_renda_atividade

Preparacao da base UFPA renda/atividade para modelagem de ML (PU Learning).


## 1. Introdução

Esta base reúne variáveis de renda e atividade produtiva em nível **família**.

O objetivo é gerar um dataset limpo, consistente e eficiente para modelagem de ML, com padronização de tipos e validações de coerência entre indicadores de renda.

## 2. Imports e Configuração do Ambiente

Importamos bibliotecas essenciais e definimos caminhos de entrada e saída.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DEFAULT_INPUT_PATH = (
    ROOT
    / 'dados'
    / 'dados_brutos'
    / 'a_tcc_v2_ufpa_renda_atividade_compilado.parquet'
)
INPUT_PATH = Path(os.getenv('RENDA_ATIVIDADE_PATH', str(DEFAULT_INPUT_PATH)))
OUTPUT_PATH = (
    ROOT
    / 'dados'
    / 'dados_processados'
    / 'df_caf_ufpa_renda_atividade.parquet'
)

# Use fração apenas para desenvolvimento rápido. Em produção, deixe None.
SAMPLE_FRAC = None  # Ex.: 0.1

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Arquivo não encontrado: {INPUT_PATH}')

print(f'ROOT: {ROOT}')
print(f'INPUT_PATH: {INPUT_PATH}')
print(f'OUTPUT_PATH: {OUTPUT_PATH}')
print(f'pyarrow: {pa.__version__}')

## 3. Carregamento do Dataset

Carregamos o parquet bruto e inspecionamos rapidamente estrutura, tamanho e tipos.

In [ ]:
colunas_interesse = [
    'id_familia',
    'ibge',
    'renda_total_estimada',
    'renda_total_auferida',
    'renda_dentro_estimada',
    'renda_dentro_auferida',
    'renda_fora_estimada',
    'renda_fora_auferida',
    'n_tipos_renda',
    'n_produtos',
    'renda_top1_produto_estimada',
    'share_top1_produto_estimada',
    'tem_agroecologia',
    'tem_agroindustria',
    'tem_pecuaria',
]

df = pd.read_parquet(INPUT_PATH, engine='pyarrow', columns=colunas_interesse)

if SAMPLE_FRAC is not None:
    df = df.sample(frac=SAMPLE_FRAC, random_state=42)

memoria_inicial_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

print(f'Shape inicial: {df.shape}')
print(f'Memória inicial: {memoria_inicial_mb:,.2f} MB')
display(df.head(3))
df.info(show_counts=True)

## 4. Diagnóstico Inicial dos Dados

Nesta etapa fazemos checagens leves de missing, cardinalidade, duplicatas e distribuições principais.

In [ ]:
print('Missing por coluna (ordem decrescente):')
display(df.isna().sum().sort_values(ascending=False))

print('Cardinalidade (nunique) por coluna:')
display(df.nunique(dropna=True).sort_values())

for col in ['n_tipos_renda', 'n_produtos', 'tem_agroecologia', 'tem_agroindustria', 'tem_pecuaria']:
    if col in df.columns:
        print(f'\nDistribuição de {col}:')
        display(df[col].value_counts(dropna=False).sort_index())

if 'id_familia' in df.columns:
    duplicados_id_familia = int(df.duplicated(subset=['id_familia']).sum())
    print(f'\nDuplicatas de id_familia: {duplicados_id_familia:,}')

## 5. Limpeza Básica

Aplicamos limpeza essencial para manter consistência sem remover registros desnecessariamente:
- remoção de duplicatas por `id_familia`;
- saneamento de valores de renda negativos (se existirem);
- tratamento de `share_top1_produto_estimada` fora da faixa [0, 1].

In [ ]:
linhas_antes_limpeza = len(df)

# 5.1) Remoção de duplicatas por chave administrativa (id_familia)
if 'id_familia' in df.columns:
    n_dup = int(df.duplicated(subset=['id_familia']).sum())
    if n_dup > 0:
        df = df.drop_duplicates(subset=['id_familia'], keep='first').copy()
    print(f'Duplicatas removidas por id_familia: {n_dup:,}')

# 5.2) Conversão explêcita de colunas numéricas de renda
colunas_renda = [
    'renda_total_estimada',
    'renda_total_auferida',
    'renda_dentro_estimada',
    'renda_dentro_auferida',
    'renda_fora_estimada',
    'renda_fora_auferida',
    'renda_top1_produto_estimada',
]

for col in colunas_renda:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        n_neg = int((df[col] < 0).sum())
        if n_neg > 0:
            df.loc[df[col] < 0, col] = np.nan
        print(f'{col}: negativos tratados como NaN = {n_neg:,}')

# 5.3) Share deve estar em [0, 1]
if 'share_top1_produto_estimada' in df.columns:
    df['share_top1_produto_estimada'] = pd.to_numeric(
        df['share_top1_produto_estimada'],
        errors='coerce',
    )
    share_col = 'share_top1_produto_estimada'
    share_invalido = df[share_col].notna() & ~df[share_col].between(0, 1)
    n_share_invalido = int(share_invalido.sum())
    if n_share_invalido > 0:
        df.loc[share_invalido, share_col] = np.nan
    print(f'share fora de [0, 1] tratados como NaN: {n_share_invalido:,}')

# 5.4) Feature auxiliar de presença de renda
if 'renda_total_estimada' in df.columns:
    df['st_possui_renda'] = (df['renda_total_estimada'] > 0).astype('int8')

print(f'Shape antes da limpeza: ({linhas_antes_limpeza:,}, {len(colunas_interesse)})')
print(f'Shape após limpeza: ({df.shape[0]:,}, {df.shape[1]})')

## 6. Padronização de Tipos de Dados

In [ ]:
# 6.1) Identificadores em string
for col in ['id_familia', 'ibge']:
    if col in df.columns:
        df[col] = df[col].astype('string')

# Garante padrão do código IBGE com 7 dígitos
if 'ibge' in df.columns:
    df['ibge'] = df['ibge'].str.zfill(7)

# 6.2) Contagens e flags em int8
int8_cols = [
    'n_tipos_renda',
    'n_produtos',
    'tem_agroecologia',
    'tem_agroindustria',
    'tem_pecuaria',
    'st_possui_renda',
]

for col in int8_cols:
    if col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype('int8')
        else:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].isna().any():
                raise ValueError(f'{col} contém valores inválidos para cast em int8.')
            df[col] = df[col].astype('int8')

# 6.3) Variáveis contínuas em float32
float32_cols = [
    'renda_total_estimada',
    'renda_total_auferida',
    'renda_dentro_estimada',
    'renda_dentro_auferida',
    'renda_fora_estimada',
    'renda_fora_auferida',
    'renda_top1_produto_estimada',
    'share_top1_produto_estimada',
]

for col in float32_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

print('Dtypes após padronização:')
display(df.dtypes)

## 7. Tratamento de Missing

Aplicamos preenchimento apenas para `share_top1_produto_estimada` quando não há produtos/renda, preservando ausências que indiquem inconsistência.

In [ ]:
if {'share_top1_produto_estimada', 'n_produtos', 'renda_total_estimada'}.issubset(df.columns):
    cond_fill_share = (
        df['share_top1_produto_estimada'].isna()
        & (df['n_produtos'] == 0)
        & (df['renda_total_estimada'] == 0)
    )
    n_fill_share = int(cond_fill_share.sum())
    if n_fill_share > 0:
        df.loc[cond_fill_share, 'share_top1_produto_estimada'] = np.float32(0.0)
    print(f'share preenchido com 0.0 (sem produtos e sem renda): {n_fill_share:,}')

print('Missing após tratamento (ordem decrescente):')
display(df.isna().sum().sort_values(ascending=False))

## 8. Otimização de Memória

Como os casts foram feitos de forma explêcita no passo anterior, não aplicamos downcast automático adicional.

Aqui medimos apenas o ganho de memória alcançado.

In [ ]:
memoria_final_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
reducao_pct = ((memoria_inicial_mb - memoria_final_mb) / memoria_inicial_mb) * 100

print(f'Memória inicial: {memoria_inicial_mb:,.2f} MB')
print(f'Memória final:   {memoria_final_mb:,.2f} MB')
print(f'Redução:         {reducao_pct:,.2f}%')

## 9. Validação Final

Verificamos estrutura final, coerência entre variáveis de renda e possíveis valores impossíveis restantes.

In [ ]:
print(f'Shape final (pré-reset): {df.shape}')

df.info(show_counts=True)
display(df.head(5))

In [ ]:
# Coerência dos totais de renda (ignora NaN)
# Tolerância em unidades monetárias para evitar falso-positivo por precisão float32.
tolerancia = 10.0

if {
    'renda_total_estimada',
    'renda_dentro_estimada',
    'renda_fora_estimada',
}.issubset(df.columns):
    inconsist_est = int(
        (~np.isclose(
            df['renda_total_estimada'],
            df['renda_dentro_estimada'] + df['renda_fora_estimada'],
            atol=tolerancia,
            rtol=0,
            equal_nan=True,
        )).sum()
    )
    print(f'Inconsistências total_estimada vs dentro+fora (tol={tolerancia}): {inconsist_est:,}')

if {
    'renda_total_auferida',
    'renda_dentro_auferida',
    'renda_fora_auferida',
}.issubset(df.columns):
    inconsist_auf = int(
        (~np.isclose(
            df['renda_total_auferida'],
            df['renda_dentro_auferida'] + df['renda_fora_auferida'],
            atol=tolerancia,
            rtol=0,
            equal_nan=True,
        )).sum()
    )
    print(f'Inconsistências total_auferida vs dentro+fora (tol={tolerancia}): {inconsist_auf:,}')

if {'share_top1_produto_estimada', 'renda_top1_produto_estimada', 'renda_total_estimada'}.issubset(df.columns):
    share_fora_faixa = int(
        (
            df['share_top1_produto_estimada'].notna()
            & ~df['share_top1_produto_estimada'].between(0, 1)
        ).sum()
    )
    top1_maior_total = int(
        (df['renda_top1_produto_estimada'] > df['renda_total_estimada']).sum()
    )
    print(f'Share fora de [0, 1] após limpeza: {share_fora_faixa:,}')
    print(f'Casos com renda_top1_produto_estimada > renda_total_estimada: {top1_maior_total:,}')


In [ ]:
for col in ['tem_agroecologia', 'tem_agroindustria', 'tem_pecuaria', 'st_possui_renda']:
    if col in df.columns:
        print(f'\nDistribuição de {col}:')
        display(df[col].value_counts(dropna=False).sort_index())

## 10. Reset do índice

Após remoções e filtros, resetamos o índice para manter continuidade e evitar índices quebrados.

In [ ]:
df = df.reset_index(drop=True)
print(f'Shape final (pós-reset): {df.shape}')
print(f'índice contínuo: {df.index.min()} até {df.index.max()}')

## 11. Persistência do Dataset

Salvamos a versão limpa em parquet para uso nas próximas etapas do pipeline.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False)

print(f'Arquivo salvo em: {OUTPUT_PATH}')
print(f'Shape final salvo: {df.shape}')
print(f'Total de colunas: {len(df.columns)}')
print('Colunas disponíveis:')
display(df.columns.to_list())